# ARQWELIA Lot 2 — SDXL Inpainting on a free GPU (benchmark only)

**WARNING**
- GPU availability is NOT guaranteed on free environments.
- A free environment is NOT appropriate for Production.
- Delete temporary files after the session.
- Do NOT use a real user photo during Phase 0A (synthetic benchmark images only).
- Do NOT expose ComfyUI on the Internet; no public tunnel is included.
- Do NOT store any DeepSeek API key in this notebook.
- **Privacy**: no image-generation API is called. In a hosted notebook
  environment such as Kaggle, the source image and mask are processed on the
  hosting provider's infrastructure. Use synthetic benchmark data only.
- Never use real user photos in Phase 0A.

## 1. GPU

In [ ]:
import subprocess
try:
    out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True, timeout=20)
    print(out.stdout or "NO GPU / nvidia-smi not found")
except Exception as exc:
    print("GPU check failed:", exc)

## 2. Configuration

In [ ]:
import os
import time
import json
import hashlib

MODEL_ID = "diffusers/stable-diffusion-xl-1.0-inpainting-0.1"
MODEL_REVISION = "115134f363124c53c7d878647567d04daf26e41e"
SOURCE_PATH = "dataset/photos/synthetic01.png"
MASK_PATH = "dataset/masks/synthetic01-pool-mask.png"
SOURCE_EXPECTED_SHA256 = "fe52d460e8bf9180ba7fd96b3d860d3dbac4d3103ce57957ea35a1a13d97d467"
MASK_EXPECTED_SHA256 = "d9d3f1a947fd6ade465f3da7bd6943a97dbfc871e37bd40f983c2b4f42ce032e"
WORKING = 1024
OUT_DIR = "benchmark-out/deepseek-comfyui-poc"

# --- single-use gate (Round 6) ---
AUTHORIZED_GENERATIONS = 1
generation_attempts = 0

# --- SECOND benchmark (POC experimental, prepared but NOT executed here) ---
# First run visually failed: mask regenerated as grass, no pool; prompt was
# truncated (95 tokens > CLIP 77). These params are experimental POC values.
POC_PARAMS = {
    "seed": 43,
    "configuredInferenceSteps": 35,
    "guidance_scale": 8.0,
    "strength": 0.99,
    "padding_mask_crop": 64,
    "note": "POC experimental — tune after a manual visual check",
}

# --- quality gate ---
generationStatus = "pending"
visualAcceptance = "pending"
rejectionReason = None

# --- error sanitizer (defined before any generation can use it) ---
def sanitize_generation_error(exc):
    import re
    text = str(exc)
    text = re.sub(r"(sk-|nvapi-|hf_)[A-Za-z0-9_\-]+", r"\1[REDACTED]", text)
    text = re.sub(r"(?i)\bBearer\s+[A-Za-z0-9._~+/=-]+", "Bearer [REDACTED]", text)
    text = re.sub(r"(?i)([?&](?:token|access_token|api_key)=)[^&\s]+", r"\1[REDACTED]", text)
    return text[:1000]

print("MODEL_ID:", MODEL_ID)
print("MODEL_REVISION:", MODEL_REVISION)
print("AUTHORIZED_GENERATIONS:", AUTHORIZED_GENERATIONS)
assert MODEL_REVISION is not None and len(MODEL_REVISION) > 0, "MODEL_REVISION must be set (pinned) before the official run"


## 3. Dependencies

In [ ]:
import sys
!{sys.executable} -m pip install --quiet torch==2.4.1 diffusers==0.31.0 transformers==4.44.2 accelerate==0.34.2 pillow==10.4.0 numpy==1.26.4
print("dependencies installed")

## 3b. Source + mask SHA precheck (BEFORE model load)

In [ ]:
from PIL import Image

def file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

source_file_sha = file_sha256(SOURCE_PATH)
assert source_file_sha == SOURCE_EXPECTED_SHA256, (
    f"Source SHA-256 mismatch: got {source_file_sha}, expected {SOURCE_EXPECTED_SHA256} — STOP"
)
mask_file_sha = file_sha256(MASK_PATH)
assert mask_file_sha == MASK_EXPECTED_SHA256, (
    f"Mask SHA-256 mismatch: got {mask_file_sha}, expected {MASK_EXPECTED_SHA256} — STOP"
)

img_check = Image.open(SOURCE_PATH).convert("RGB")
assert img_check.format is not None, "Source not decodable"
assert img_check.size == (1536, 1024), f"Source dims {img_check.size} != 1536x1024"
# Non-uniform source check (luma variance).
import numpy as np
_a = np.asarray(img_check).astype(int)
_rng = int(_a.max(axis=(0,1)).mean() - _a.min(axis=(0,1)).mean())
assert _rng > 15, "Source image is uniform (no real content)"
print("source sha256:", source_file_sha)
print("mask sha256:", mask_file_sha)
print("source dims:", img_check.size, "| non-uniform: yes")
# Any failure above sets generation_attempts = 0 (never consumed here).
generation_attempts = 0


## 4. Model

In [ ]:
import torch
from diffusers import StableDiffusionXLInpaintPipeline

# GPU gate: only CUDA availability is required (a Tesla T4 is accepted).
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU required for this benchmark")

pipe = StableDiffusionXLInpaintPipeline.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True,
)

pipe.enable_model_cpu_offload()
# Non-deprecated VAE API (Round 6):
pipe.vae.enable_slicing()
pipe.vae.enable_tiling()

gpu = torch.cuda.get_device_name(0)
props = torch.cuda.get_device_properties(0)
print("GPU:", gpu)
print("VRAM total (GB):", round(props.total_memory / 1024**3, 2))
print("memory mode: model_cpu_offload")
print("revision:", MODEL_REVISION)
print("dtype:", pipe.dtype)

## 5. Source + mask + canvas

In [ ]:
from PIL import Image

img = Image.open(SOURCE_PATH).convert("RGB")
mask = Image.open(MASK_PATH).convert("L")
orig_w, orig_h = img.size
assert mask.size == img.size, "mask and image must be the same size"

scale = min(WORKING / orig_w, WORKING / orig_h)
resized_w = max(1, round(orig_w * scale))
resized_h = max(1, round(orig_h * scale))
offset_x = max(0, (WORKING - resized_w) // 2)
offset_y = max(0, (WORKING - resized_h) // 2)

working_image = Image.new("RGB", (WORKING, WORKING), (0, 0, 0))
working_image.paste(img.resize((resized_w, resized_h)), (offset_x, offset_y))
working_mask = Image.new("L", (WORKING, WORKING), 0)
working_mask.paste(mask.resize((resized_w, resized_h), Image.NEAREST), (offset_x, offset_y))

mapping = {
    "scale": scale,
    "offsetX": offset_x,
    "offsetY": offset_y,
    "resizedWidth": resized_w,
    "resizedHeight": resized_h,
    "originalWidth": orig_w,
    "originalHeight": orig_h,
    "workingWidth": WORKING,
    "workingHeight": WORKING,
}
print("working image size:", working_image.size, "| mask size:", working_mask.size)

## 6. Prompts (CLIP-safe, both SDXL tokenizers, fail-closed)

In [ ]:
import json

POSITIVE_PROMPT = (
    "Large photorealistic rectangular in-ground swimming pool, clear blue water, "
    "natural limestone coping, correctly embedded in this lawn, realistic perspective "
    "and sunlight, Mediterranean residential garden."
)
NEGATIVE_PROMPT = (
    "grass inside pool, empty lawn, pond, people, text, logo, distorted house, "
    "warped geometry, extra pool, artificial reflections."
)

def clip_token_counts(text):
    if pipe.tokenizer is None or pipe.tokenizer_2 is None:
        raise RuntimeError("Both SDXL tokenizers are required")
    count_1 = len(pipe.tokenizer(text, return_tensors="pt", padding=False, truncation=False).input_ids[0])
    count_2 = len(pipe.tokenizer_2(text, return_tensors="pt", padding=False, truncation=False).input_ids[0])
    return count_1, count_2

# Fail closed: any tokenizer exception or count > 75 stops BEFORE pipe().
positiveTokenizer1Count, positiveTokenizer2Count = clip_token_counts(POSITIVE_PROMPT)
negativeTokenizer1Count, negativeTokenizer2Count = clip_token_counts(NEGATIVE_PROMPT)
assert positiveTokenizer1Count <= 75, f"positive tokenizer_1 too long: {positiveTokenizer1Count}"
assert positiveTokenizer2Count <= 75, f"positive tokenizer_2 too long: {positiveTokenizer2Count}"
assert negativeTokenizer1Count <= 75, f"negative tokenizer_1 too long: {negativeTokenizer1Count}"
assert negativeTokenizer2Count <= 75, f"negative tokenizer_2 too long: {negativeTokenizer2Count}"

visual_brief = {
    "version": "arqwelia-visual-brief-v1",
    "concept": "A",
    "sceneType": "residential_garden_pool_inpainting",
    "pool": {"shape": "rectangular", "estimatedDimensions": "8x4m", "placement": "central_open_lawn", "orientation": "parallel_to_house"},
    "preserve": ["house_architecture", "camera_perspective", "boundary_fences", "mature_trees", "unmasked_pixels"],
    "add": ["realistic_in_ground_pool", "natural_stone_coping", "mediterranean_landscaping"],
    "negative": ["people", "text", "logo", "house_distortion", "extra_buildings", "duplicate_pool", "floating_objects", "unrealistic_reflections"],
    "inpaintingPrompt": POSITIVE_PROMPT,
    "negativePrompt": NEGATIVE_PROMPT,
    "recommended": {"steps": POC_PARAMS["configuredInferenceSteps"], "cfg": POC_PARAMS["guidance_scale"], "strength": POC_PARAMS["strength"], "seed": POC_PARAMS["seed"]},
}
print("visual brief ready:", visual_brief["version"], visual_brief["concept"])
print("positive tok1/tok2:", positiveTokenizer1Count, positiveTokenizer2Count)
print("negative tok1/tok2:", negativeTokenizer1Count, negativeTokenizer2Count)

## 7. Effective inference steps (distinct from configured)

In [ ]:
configuredInferenceSteps = POC_PARAMS["configuredInferenceSteps"]
strength = POC_PARAMS["strength"]
effectiveInferenceSteps = min(
    int(configuredInferenceSteps * strength),
    configuredInferenceSteps,
)
print("configuredInferenceSteps:", configuredInferenceSteps)
print("strength:", strength)
print("effectiveInferenceSteps:", effectiveInferenceSteps)

## 8. Single generation (single-use gate, honest failure report)

In [ ]:
import torch

# All pre-generation work (imports, CUDA gate, env checks, tokenizer checks,
# source/mask SHA) already happened BEFORE this cell. The attempt is consumed
# only immediately before pipe() and inside the same try block.

generationStatus = "in_progress"
generation_started_at = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
gen_started = time.time()
generator = torch.Generator(device="cuda" if torch.cuda.is_available() else "cpu").manual_seed(visual_brief["recommended"]["seed"])

try:
    if generation_attempts >= AUTHORIZED_GENERATIONS:
        raise RuntimeError("Owner generation limit reached")
    generation_attempts += 1
    result = pipe(
        prompt=visual_brief["inpaintingPrompt"],
        negative_prompt=visual_brief["negativePrompt"],
        image=working_image,
        mask_image=working_mask,
        width=WORKING,
        height=WORKING,
        num_inference_steps=visual_brief["recommended"]["steps"],
        guidance_scale=visual_brief["recommended"]["cfg"],
        strength=visual_brief["recommended"]["strength"],
        padding_mask_crop=POC_PARAMS["padding_mask_crop"],
        generator=generator,
    )
    generationStatus = "succeeded"
except Exception as exc:
    generationStatus = "failed"
    generationError = sanitize_generation_error(exc)
    os.makedirs(OUT_DIR, exist_ok=True)
    failure_report = {
        "modelId": MODEL_ID,
        "modelRevision": MODEL_REVISION,
        "GPU": gpu,
        "VRAM": round(props.total_memory / 1024**3, 2),
        "sourceSha256": source_file_sha,
        "sourceExpectedSha256": SOURCE_EXPECTED_SHA256,
        "sourceSha256Verified": True,
        "maskSha256": mask_file_sha,
        "generationAttempts": generation_attempts,
        "retryExecuted": False,
        "startedAt": generation_started_at,
        "durationSeconds": round(time.time() - gen_started, 2),
        "status": "failed",
        "generationStatus": "failed",
        "generationError": generationError,
        "visualAcceptance": visualAcceptance,
        "rejectionReason": rejectionReason,
    }
    with open(os.path.join(OUT_DIR, "notebook-run-report.json"), "w") as f:
        json.dump(failure_report, f, indent=2)
    raise

generated = result.images[0].convert("RGB")
print("generation done")


## 9. Recomposition / restoration

In [ ]:
from PIL import ImageChops

canvas_composite = Image.composite(
    generated,
    working_image,
    working_mask,
)

crop_box = (
    offset_x,
    offset_y,
    offset_x + resized_w,
    offset_y + resized_h,
)

cropped = canvas_composite.crop(crop_box).resize(
    (orig_w, orig_h),
    Image.LANCZOS,
)

mask_orig = mask.resize(
    (orig_w, orig_h),
    Image.NEAREST,
)

final_output = Image.composite(
    cropped,
    img,
    mask_orig,
)

assert final_output.size == (1536, 1024), f"final output size {final_output.size} != (1536, 1024)"
print("final output size:", final_output.size)

## 10. Metrics

In [ ]:
import numpy as np

# Raw model output vs working canvas source.
src_arr = np.asarray(working_image.convert("RGB")).astype(int)
gen_arr = np.asarray(generated.convert("RGB")).astype(int)
mask_arr = np.asarray(working_mask.convert("L")) >= 128
raw_diff = np.abs(src_arr - gen_arr).sum(axis=2) > 12
rawGeneratedChangedPixelRatioInsideMask = float(
    (raw_diff & mask_arr).sum() / max(mask_arr.sum(), 1)
)
rawGeneratedUnchangedPixelRatioOutsideMask = float(
    (~raw_diff & ~mask_arr).sum() / max((~mask_arr).sum(), 1)
)

# Final composite vs ORIGINAL source outside the mask (deterministic => 1.0).
final_src = np.asarray(img.convert("RGB")).astype(int)
final_out = np.asarray(final_output.convert("RGB")).astype(int)
mask_orig_arr = np.asarray(mask_orig.convert("L")) >= 128
final_diff = np.abs(final_src - final_out).sum(axis=2) > 12
finalCompositeUnchangedPixelRatioOutsideMask = float(
    (~final_diff & ~mask_orig_arr).sum() / max((~mask_orig_arr).sum(), 1)
)
print("rawGeneratedChangedPixelRatioInsideMask:", round(rawGeneratedChangedPixelRatioInsideMask, 4))
print("rawGeneratedUnchangedPixelRatioOutsideMask:", round(rawGeneratedUnchangedPixelRatioOutsideMask, 4))
print("finalCompositeUnchangedPixelRatioOutsideMask:", round(finalCompositeUnchangedPixelRatioOutsideMask, 4))

## 11. Save + report

In [ ]:
import hashlib, os, time, json
os.makedirs(OUT_DIR, exist_ok=True)

def file_sha256(path):
    """SHA-256 of the actual PNG file bytes."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def pixel_sha256(img):
    return hashlib.sha256(img.tobytes()).hexdigest()

final_path = os.path.join(OUT_DIR, "notebook-sdxl-final.png")
canvas_path = os.path.join(OUT_DIR, "notebook-sdxl-canvas.png")
canvas_composite.save(canvas_path)
final_output.save(final_path)
with open(os.path.join(OUT_DIR, "notebook-mapping.json"), "w") as f:
    json.dump(mapping, f, indent=2)

completed_at = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
duration_seconds = round(time.time() - gen_started, 2)
report = {
    "modelId": MODEL_ID,
    "modelRevision": MODEL_REVISION,
    "GPU": gpu,
    "VRAM": round(props.total_memory / 1024**3, 2),
    "sourceSha256": file_sha256(SOURCE_PATH),
    "maskSha256": mask_file_sha,
    "sourceSha256": source_file_sha,
    "sourceExpectedSha256": SOURCE_EXPECTED_SHA256,
    "sourceSha256Verified": True,
    "generationAttempts": generation_attempts,
    "retryExecuted": False,
    "seed": visual_brief["recommended"]["seed"],
    "steps": visual_brief["recommended"]["steps"],
    "cfg": visual_brief["recommended"]["cfg"],
    "strength": visual_brief["recommended"]["strength"],
    "configuredInferenceSteps": configuredInferenceSteps,
    "effectiveInferenceSteps": effectiveInferenceSteps,
    "paddingMaskCrop": POC_PARAMS["padding_mask_crop"],
    "promptPositiveTokenCount": max(positiveTokenizer1Count, positiveTokenizer2Count),
    "promptNegativeTokenCount": max(negativeTokenizer1Count, negativeTokenizer2Count),
    "positiveTokenizer1Count": positiveTokenizer1Count,
    "positiveTokenizer2Count": positiveTokenizer2Count,
    "negativeTokenizer1Count": negativeTokenizer1Count,
    "negativeTokenizer2Count": negativeTokenizer2Count,
    "rawGeneratedChangedPixelRatioInsideMask": rawGeneratedChangedPixelRatioInsideMask,
    "rawGeneratedUnchangedPixelRatioOutsideMask": rawGeneratedUnchangedPixelRatioOutsideMask,
    "finalCompositeUnchangedPixelRatioOutsideMask": finalCompositeUnchangedPixelRatioOutsideMask,
    "workingDimensions": [canvas_composite.size[0], canvas_composite.size[1]],
    "finalDimensions": [final_output.size[0], final_output.size[1]],
    "workingFileSha256": file_sha256(canvas_path),
    "finalFileSha256": file_sha256(final_path),
    "finalPixelSha256": pixel_sha256(final_output),
    "startedAt": generation_started_at,
    "completedAt": completed_at,
    "durationSeconds": duration_seconds,
    "status": "succeeded",
    "generationStatus": generationStatus,
    "visualAcceptance": visualAcceptance,
    "rejectionReason": rejectionReason,
}
with open(os.path.join(OUT_DIR, "notebook-run-report.json"), "w") as f:
    json.dump(report, f, indent=2)
print("final file sha256:", report["finalFileSha256"])
print("report written")


## 12. Stop

In [ ]:
print("seed:", visual_brief["recommended"]["seed"])
print("configuredInferenceSteps:", configuredInferenceSteps)
print("effectiveInferenceSteps:", effectiveInferenceSteps)
print("cfg:", visual_brief["recommended"]["cfg"])
print("strength:", visual_brief["recommended"]["strength"])
print("padding_mask_crop:", POC_PARAMS["padding_mask_crop"])
print("working canvas:", canvas_composite.size)
print("final output size:", final_output.size)
print("model:", MODEL_ID, "| revision:", MODEL_REVISION)
print("generationStatus:", generationStatus)
print("visualAcceptance:", visualAcceptance)
print("rejectionReason:", rejectionReason)